In [2]:
import cv2
import numpy as np
from collections import deque
from tkinter import Tk
from tkinter.filedialog import askopenfilename

Tk().withdraw()

source_name = input("For file search, press enter or 0 for webcam: ")

if source_name == "0":
    source_name = 0
else:
    source_name = askopenfilename(
        title="Select video file",
        filetypes=[("MP4 files", "*.mp4"), ("All files", "*.*")]
    )
    if not source_name:
        exit()

MIN_AREA = 100
TRACK_LENGTH = 10

positions = deque(maxlen=TRACK_LENGTH)

cap = cv2.VideoCapture(source_name)
fps = cap.get(cv2.CAP_PROP_FPS)
dt = 1.0 / fps
frame_idx = 0

bg = cv2.createBackgroundSubtractorMOG2(
    history=200,
    varThreshold=50,
    detectShadows=True
)

def initial_speed_ok(pts, v_min=80):
    if len(pts) < 3:
        return False

    pts = np.array(pts)
    t = pts[:,0]
    x = pts[:,1]
    y = pts[:,2]

    vx = np.gradient(x, t)
    vy = np.gradient(y, t)

    v = np.hypot(vx, vy)
    return np.max(v[:3]) > v_min

def is_projectile(pts):
    if len(pts) < TRACK_LENGTH:
        return False

    pts = np.array(pts)
    t = pts[:,0]
    x = pts[:,1]
    y = pts[:,2]

    vx = np.gradient(x, t) # sízszintes sebesség
    vx_std = np.std(vx) # szórás
    vx_mean = np.mean(np.abs(vx)) # sebesség átlagos nagysága

    if vx_std > 0.3 * vx_mean:
        return False

    vy = np.gradient(y, t) # függőleges sebesség
    coeffs = np.polyfit(t, vy, 1)
    vy_fit = np.polyval(coeffs, t)
    err = np.mean(np.abs(vy - vy_fit))

    if err > 0.4 * np.mean(np.abs(vy)):
        return False

    return True

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx < 10:
        fg = bg.apply(frame, learningRate=0.5)
    else:
        fg = bg.apply(frame, learningRate=0.01)

    fg = cv2.medianBlur(fg, 5)
    _, fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        c = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(c)

        if area > MIN_AREA:
            x, y, w, h = cv2.boundingRect(c)
            cx, cy = x + w // 2, y + h // 2

            t = frame_idx * dt

            accept = True

            if len(positions) > 0:
                _, px, py = positions[-1]
                dist = np.hypot(cx - px, cy - py)

                if dist > 250:
                    accept = False
            if accept:
                positions.append((t, cx, cy))

                cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
                cv2.circle(frame, (cx,cy), 4, (0,0,255), -1)


    if is_projectile(positions) and initial_speed_ok(positions):
        for i in range(1, len(positions)):
            cv2.line(frame, (positions[i-1][1], positions[i-1][2]), (positions[i][1], positions[i][2]), (0,0,255), 2)

        cv2.putText(
            frame,
            "ELHAJITOTT TEST",
            (30,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0,0,255),
            2
        )
    # cv2.imshow("FG MASK", fg)

    cv2.imshow("Projectile Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_idx += 1

cap.release()
cv2.destroyAllWindows()